In [1]:
import numpy as np
import matplotlib.pyplot as plt

from dls.neural_net import predict, train
from dls.dataset import boston_train_test_split
from dls.model_error import mae, rmse

from dls.dnn import (
    Dense,
    Linear,
    MeanSquaredError,
    NeuralNetwork,
    SGD,
    Sigmoid,
    Trainer,
)
from dls.dnn import eval_regression_model


In [2]:
boston_ns = boston_train_test_split()
X_train, y_train, X_test, y_test, features = (
    boston_ns.X_train,
    boston_ns.y_train,
    boston_ns.X_test,
    boston_ns.y_test,
    boston_ns.features,
)


In [3]:
deep_neural_network = NeuralNetwork(
    layers=[
        Dense(neurons=13, activation=Sigmoid()),
        Dense(neurons=13, activation=Sigmoid()),
        Dense(neurons=1, activation=Linear()),
    ],
    loss=MeanSquaredError(),
    learning_rate=0.01,
    seed=1,
)

sgd = SGD(learning_rate=0.01)
sgd.net = deep_neural_network
trainer = Trainer(net=deep_neural_network, optim=sgd)
trainer.fit(X_train, y_train, X_test, y_test, epochs=50, eval_every=10, seed=20190501)
print()
# generates predictions
eval_regression_model(deep_neural_network, X_test, y_test)


Epoch 10: loss = 33.927
Epoch 20: loss = 20.639
Epoch 30: loss = 18.125
Epoch 40: loss = 17.204
Epoch 50: loss = 14.189

Mean absolute error: 2.40

Root mean squared error 3.77


# Deep Neural Network Extensions

In [2]:
from dls.dataset import mnist_train_test_split
from dls.model_error import calc_accuracy_model
from dls.dnn_ext import SoftmaxCrossEntropyLoss
from dls.dnn_ext import SGDMomentum
from dls.dnn_ext import Tanh

RANDOM_SEED = 190119

In [3]:
mtts = mnist_train_test_split()
X_train, y_train, X_test, y_test = mtts.X_train, mtts.y_train, mtts.X_test, mtts.y_test

In [4]:
num_labels = len(y_train)
num_labels

60000

In [5]:
# one-hot encode
num_labels = len(y_train)
train_labels = np.zeros((num_labels, 10))
for i in range(num_labels):
    train_labels[i][y_train[i]] = 1

num_labels = len(y_test)
test_labels = np.zeros((num_labels, 10))
for i in range(num_labels):
    test_labels[i][y_test[i]] = 1

## MNIST Demos

### Scale data to mean 0, variance 1

In [6]:
X_train, X_test = X_train - np.mean(X_train), X_test - np.mean(X_train)
np.min(X_train), np.max(X_train), np.min(X_test), np.max(X_test)

(np.float64(-33.318421449829934),
 np.float64(221.68157855017006),
 np.float64(-33.318421449829934),
 np.float64(221.68157855017006))

In [7]:
X_train, X_test = X_train / np.std(X_train), X_test / np.std(X_train)
np.min(X_train), np.max(X_train), np.min(X_test), np.max(X_test)

(np.float64(-0.424073894391566),
 np.float64(2.821543345689335),
 np.float64(-0.424073894391566),
 np.float64(2.821543345689335))

### Sigmoid activation

In [10]:
model = NeuralNetwork(
    layers=[
        Dense(
            neurons=89,
            activation=Sigmoid(),
        ),
        Dense(
            neurons=10,
            activation=Sigmoid(),
        ),
    ],
    loss=MeanSquaredError(),
    seed=RANDOM_SEED,
)

sgd_optimizer = SGD(learning_rate=0.1)
sgd_optimizer.net = model
trainer = Trainer(net=model, optim=sgd_optimizer)
trainer.fit(
    X_train,
    train_labels,
    X_test,
    test_labels,
    epochs=50,
    eval_every=5,
    seed=RANDOM_SEED,
    batch_size=60,
)


Epoch 5: loss = 0.836
Epoch 10: loss = 0.718
Epoch 15: loss = 0.659
Epoch 20: loss = 0.638
Epoch 25: loss = 0.627
Epoch 30: loss = 0.619
Epoch 35: loss = 0.558
Epoch 40: loss = 0.506
Epoch 45: loss = 0.499
Epoch 50: loss = 0.495


In [12]:
calc_accuracy_model(model, X_test, y_test)

The model validation accuracy is: 57.16%


#### Normalized MSE

In [13]:
model = NeuralNetwork(
    layers=[
        Dense(
            neurons=89,
            activation=Sigmoid(),
        ),
        Dense(
            neurons=10,
            activation=Sigmoid(),
        ),
    ],
    loss=MeanSquaredError(normalize=True),
    seed=RANDOM_SEED,
)

sgd_optimizer = SGD(learning_rate=0.1)
sgd_optimizer.net = model
trainer = Trainer(net=model, optim=sgd_optimizer)
trainer.fit(
    X_train,
    train_labels,
    X_test,
    test_labels,
    epochs=50,
    eval_every=5,
    seed=RANDOM_SEED,
    batch_size=60,
)

calc_accuracy_model(model, X_test, y_test)

Epoch 5: loss = 0.573
Loss increased after epoch 10, final loss was 0.573, using the model from epoch 5
The model validation accuracy is: 62.54%


### Softmax Cross Entropy Loss

In [10]:
model = NeuralNetwork(
    layers=[
        Dense(
            neurons=89,
            activation=Sigmoid(),
        ),
        Dense(
            neurons=10,
            activation=Linear(),
        ),
    ],
    loss=SoftmaxCrossEntropyLoss(),
    seed=RANDOM_SEED,
)

sgd_optimizer = SGD(learning_rate=0.1)
sgd_optimizer.net = model
trainer = Trainer(net=model, optim=sgd_optimizer)
trainer.fit(
    X_train,
    train_labels,
    X_test,
    test_labels,
    epochs=50,
    eval_every=5,
    seed=RANDOM_SEED,
    batch_size=60,
)
print()
calc_accuracy_model(model, X_test, y_test)

Epoch 5: loss = 0.719
Epoch 10: loss = 0.611
Epoch 15: loss = 0.559
Epoch 20: loss = 0.530
Epoch 25: loss = 0.505
Epoch 30: loss = 0.488
Epoch 35: loss = 0.475
Epoch 40: loss = 0.467
Epoch 45: loss = 0.459
Epoch 50: loss = 0.453

The model validation accuracy is: 92.61%


#### SGD with Momentum

In [11]:
model = NeuralNetwork(
    layers=[
        Dense(
            neurons=89,
            activation=Sigmoid(),
        ),
        Dense(
            neurons=10,
            activation=Linear(),
        ),
    ],
    loss=SoftmaxCrossEntropyLoss(),
    seed=RANDOM_SEED,
)

sgd_optimizer = SGDMomentum(learning_rate=0.1, momentum=0.9)
sgd_optimizer.net = model
trainer = Trainer(net=model, optim=sgd_optimizer)
trainer.fit(
    X_train,
    train_labels,
    X_test,
    test_labels,
    epochs=50,
    eval_every=5,
    seed=RANDOM_SEED,
    batch_size=60,
)
print()
calc_accuracy_model(model, X_test, y_test)


Epoch 5: loss = 0.414
Epoch 10: loss = 0.355
Loss increased after epoch 15, final loss was 0.355, using the model from epoch 10

The model validation accuracy is: 94.25%


### Linear learning rate decay

In [11]:
model = NeuralNetwork(
    layers=[
        Dense(
            neurons=89,
            activation=Sigmoid(),
        ),
        Dense(
            neurons=10,
            activation=Linear(),
        ),
    ],
    loss=SoftmaxCrossEntropyLoss(),
    seed=RANDOM_SEED,
)

sgd_optimizer = SGDMomentum(
    learning_rate=0.15, momentum=0.9, final_lr=0.05, decay_type="linear"
)
sgd_optimizer.net = model
trainer = Trainer(net=model, optim=sgd_optimizer)
trainer.fit(
    X_train,
    train_labels,
    X_test,
    test_labels,
    epochs=25,
    eval_every=5,
    seed=RANDOM_SEED,
    batch_size=60,
)
print()
calc_accuracy_model(model, X_test, y_test)


Epoch 5: loss = 0.356

The model validation accuracy is: 93.95%


### Exponential learning rate decay

In [10]:
model = NeuralNetwork(
    layers=[
        Dense(
            neurons=89,
            activation=Sigmoid(),
        ),
        Dense(
            neurons=10,
            activation=Linear(),
        ),
    ],
    loss=SoftmaxCrossEntropyLoss(),
    seed=RANDOM_SEED,
)

sgd_optimizer = SGDMomentum(
    learning_rate=0.2, momentum=0.9, final_lr=0.05, decay_type="exponential"
)
sgd_optimizer.net = model
trainer = Trainer(net=model, optim=sgd_optimizer)
trainer.fit(
    X_train,
    train_labels,
    X_test,
    test_labels,
    epochs=25,
    eval_every=5,
    seed=RANDOM_SEED,
    batch_size=60,
)
print()
calc_accuracy_model(model, X_test, y_test)


Epoch 5: loss = 0.393

The model validation accuracy is: 93.46%


### Weight initialization in Layers

In [14]:
model = NeuralNetwork(
    layers=[
        Dense(neurons=89, activation=Sigmoid(), weight_init="glorot"),
        Dense(neurons=10, activation=Linear(), weight_init="glorot"),
    ],
    loss=SoftmaxCrossEntropyLoss(),
    seed=RANDOM_SEED,
)

sgd_optimizer = SGDMomentum(
    learning_rate=0.2, momentum=0.9, final_lr=0.05, decay_type="exponential"
)
sgd_optimizer.net = model
trainer = Trainer(net=model, optim=sgd_optimizer)
trainer.fit(
    X_train,
    train_labels,
    X_test,
    test_labels,
    epochs=25,
    eval_every=5,
    seed=RANDOM_SEED,
    batch_size=60,
)
print()
calc_accuracy_model(model, X_test, y_test)


Epoch 5: loss = 0.180

The model validation accuracy is: 97.14%


#### Using Tanh activation

In [8]:
model = NeuralNetwork(
    layers=[
        Dense(neurons=89, activation=Tanh(), weight_init="glorot"),
        Dense(neurons=10, activation=Linear(), weight_init="glorot"),
    ],
    loss=SoftmaxCrossEntropyLoss(),
    seed=RANDOM_SEED,
)

sgd_optimizer = SGDMomentum(
    learning_rate=0.2, momentum=0.9, final_lr=0.05, decay_type="exponential"
)
sgd_optimizer.net = model
trainer = Trainer(net=model, optim=sgd_optimizer)
trainer.fit(
    X_train,
    train_labels,
    X_test,
    test_labels,
    epochs=25,
    eval_every=5,
    seed=RANDOM_SEED,
    batch_size=60,
)
print()
calc_accuracy_model(model, X_test, y_test)


Epoch 5: loss = 0.453
Epoch 10: loss = 0.298
Epoch 15: loss = 0.284
Epoch 20: loss = 0.254
Epoch 25: loss = 0.238

The model validation accuracy is: 96.61%


## Dropout

In [8]:
model = NeuralNetwork(
    layers=[
        Dense(neurons=89, activation=Sigmoid(), weight_init="glorot", dropout=0.8),
        Dense(neurons=10, activation=Linear(), weight_init="glorot"),
    ],
    loss=SoftmaxCrossEntropyLoss(),
    seed=RANDOM_SEED,
)

sgd_optimizer = SGDMomentum(
    learning_rate=0.2, momentum=0.9, final_lr=0.05, decay_type="exponential"
)
sgd_optimizer.net = model
trainer = Trainer(net=model, optim=sgd_optimizer)
trainer.fit(
    X_train,
    train_labels,
    X_test,
    test_labels,
    epochs=25,
    eval_every=5,
    seed=RANDOM_SEED,
    batch_size=60,
)
print()
calc_accuracy_model(model, X_test, y_test, inference=True)


Epoch 5: loss = 0.177
Epoch 10: loss = 0.156
Loss increased after epoch 15, final loss was 0.156, using the model from epoch 10

The model validation accuracy is: 97.50%


In [9]:
model = NeuralNetwork(
    layers=[
        Dense(neurons=178, activation=Tanh(), weight_init="glorot", dropout=0.8),
        Dense(neurons=46, activation=Tanh(), weight_init="glorot", dropout=0.8),
        Dense(neurons=10, activation=Linear(), weight_init="glorot"),
    ],
    loss=SoftmaxCrossEntropyLoss(),
    seed=RANDOM_SEED,
)

sgd_optimizer = SGDMomentum(
    learning_rate=0.2, momentum=0.9, final_lr=0.05, decay_type="exponential"
)
sgd_optimizer.net = model
trainer = Trainer(net=model, optim=sgd_optimizer)
trainer.fit(
    X_train,
    train_labels,
    X_test,
    test_labels,
    epochs=25,
    eval_every=5,
    seed=RANDOM_SEED,
    batch_size=60,
)
print()
calc_accuracy_model(model, X_test, y_test, inference=True)


Epoch 5: loss = 0.337
Epoch 10: loss = 0.288
Epoch 15: loss = 0.236
Epoch 20: loss = 0.217
Epoch 25: loss = 0.199

The model validation accuracy is: 96.84%
